# Harness Rogue AI Agents

<a href="https://www.kaggle.com/code/addarm/harness-rogue-agents" target="_blank" rel="noopener noreferrer">
  <img src="https://www.kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle" title="Open in Kaggle">
</a>

![Can Your AI Agent Bypass Its Permissions? Testing Boundaries with Z3 and OpenShell](https://raw.githubusercontent.com/adamd1985/quant_research/refs/heads/main/images/harness_rogue_agents.png)

Imagine an agent tries to write to a forbidden GitHub repository. The sandbox blocks it, which is Good - but like a hacker, the agent tries a different route, and the next message says the file was written successfully.

That is the incident NVIDIA's OpenShell team describes in a [September 2026 dev note](https://nvidia.github.io/OpenShell-Research/dev-notes/posts/2026-09-10-learning-formal-methods-agent-policy-prover/). The agent did not steal a new credential. It reused the credential it already had, but switched from a REST request to `git-remote-https`, the helper program Git uses to communicate over HTTPS. The policy inspected REST calls, while Git spoke its own protocol. One door was guarded; another door to the same service was open.

This is a *rogue* behavior (or as some would call it, chutzpah!) that scares us as it acts outside the operator's intended boundary, not out of malicious intent.

This inspired us to write this article, which by the end of it you'd be able to:

- explain policy containment in plain language;
- read SAT, UNSAT, and UNKNOWN results without overclaiming;
- interpret an OpenShell counterexample as a model witness, not observed traffic; and
- distinguish a proved policy relationship from runtime enforcement and overall agent safety.

## Notebook Setup

We will use Python and Z3, then later calling our local `openshell-prover` binary, so version 0.1.5 must be available on `PATH` before the notebook starts.

In [ ]:
%pip install z3-solver==5.3.1.0

In [2]:
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import z3

SEED = 42
z3.set_param("smt.random_seed", SEED)
z3.set_param("sat.random_seed", SEED)
PROVER = shutil.which("openshell-prover")

print("python", sys.version.split()[0], "| z3", z3.get_version_string())
assert PROVER, "not found, the OpenShell cells will skip"

python 3.15.1 | z3 5.3.1


## Contained Policy Boundaries

A credential answers only one question: *who may authenticate?* A policy must answer a broader question: *what may this process do, through which route?*

In the article's toy problem, we identify an action as having five parts:


1. Binary: e.g. `gh` or `git-remote-https`. Different programs expose different capabilities.
1. Host: e.g.`api.github.com` or `github.com`. Two hostnames may reach the same service through different interfaces.
1. Protocol: e.g. REST or Git smart HTTP. Inspection written for one protocol may not understand another.
1. Method: e.g.`GET` or `POST`. Read and write operations should not be treated alike.
1. Path: e.g.`/repos/alice/demo`. A rule may apply to one resource but not its neighbors.

We now define two policies. 

The **boundary**, $B$, is the maximum access the operator is willing to permit. 

The **candidate**, $C$, is the complete policy proposed for the agent. The candidate is safe with respect to that boundary only when every action allowed by $C$ is also allowed by $B$. We refer to this as policy containment.

The model intentionally simplifies the real incident. The real policy allowed selected repository writes through REST inspection, and the credential was reused through another wire protocol. Our boundary permits only read access through REST, while one candidate adds an approved Git binary.

## Only One Bad Action

It is easier to understand containment by trying to break it. Instead of checking every permitted action one at a time, ask: "Can we find even one action that the candidate allows and the boundary forbids?"

The query is

$$
C(a) \land \neg B(a)
$$

the candidate allows action $a$, **and** the boundary does **not** allow $a$.

Z3 ([de Moura and Bjørner, 2008](https://doi.org/10.1007/978-3-540-78800-3_24)) searches for such an action. Because actions include strings, enumerated values, and logical conditions rather than only true/false variables, this is *Satisfiability Modulo Theories* (SMT) - a field which we suggest you read, though we will not focus on in the article.

The solver can report three outcomes:
- **SAT:** a counterexample exists. The candidate exceeds the boundary, and the returned model gives one witness.
- **UNSAT:** no counterexample exists within the model. The candidate is contained by the boundary.
- **UNKNOWN:** the solver did not establish either result. This is not a pass.

The phrase "within the model" is essential. Z3 proves a statement about the rules we encoded. It cannot detect a binary, protocol, credential behavior, or side effect that we forgot to represent - so don't forget!


Formally this can be represented through $\mathcal{A}$, which contains all modeled combinations of $\text{Binary} \times \text{Host} \times \text{Protocol} \times \text{Method} \times \text{Path}$, and let $A(P)=\{a\in\mathcal{A}:P(a)\}$ be the actions allowed by policy $P$. Then

$$
\begin{aligned}
A(C) \subseteq A(B)
&\iff \neg \exists\,a\in\mathcal{A}.\;C(a)\land\neg B(a)\\
&\iff C(a)\land\neg B(a)\text{ is UNSAT.}
\end{aligned}
$$

## FInding the Escape Route with Z3

We will compare two candidates with the same boundary, $B$ whic allows `gh` or `curl` reads `api.github.com` under `/repos/` through REST:

1. Candidate $C_1$: `gh` reads one repository through REST. UNSAT: narrower than $B$ 
1. Candidate $C_2$: Everything in $C_1$, plus `git-remote-https` reaching `github.com` through Git smart HTTP. SAT: contains an escape route 

Notice that the credential does not change. Only the route changes. The code below gives each part of an action a type, defines the three policies, and asks Z3 for a counterexample.

In [3]:
Binary, (gh, curl, git_remote_https) = z3.EnumSort("Binary", ["gh", "curl", "git_remote_https"])
Host, (api_github, github, pypi) = z3.EnumSort("Host", ["api.github.com", "github.com", "pypi.org"])
Protocol, (rest, git_http) = z3.EnumSort("Protocol", ["rest", "git_smart_http"])
Method, (GET, POST, PUT, DELETE) = z3.EnumSort("Method", ["GET", "POST", "PUT", "DELETE"])

binary = z3.Const("binary", Binary)
host = z3.Const("host", Host)
protocol = z3.Const("protocol", Protocol)
method = z3.Const("method", Method)
path = z3.String("path")
action = [binary, host, protocol, method, path]

B = z3.And(z3.Or(binary == gh, binary == curl), host == api_github, protocol == rest,
           method == GET, z3.PrefixOf(z3.StringVal("/repos/"), path))
C1 = z3.And(binary == gh, host == api_github, protocol == rest,
            method == GET, z3.PrefixOf(z3.StringVal("/repos/acme/widgets/"), path))
GIT_CLAUSE = z3.And(binary == git_remote_https, host == github, protocol == git_http)
C2 = z3.Or(C1, GIT_CLAUSE)


def exceeds(candidate, boundary):
    solver = z3.Solver()
    solver.add(candidate, z3.Not(boundary))
    return solver.check(), solver


vacuity = z3.Solver()
vacuity.add(C1)
assert vacuity.check() == z3.sat

result1, _ = exceeds(C1, B)
result2, solver2 = exceeds(C2, B)
print("C1 and not B:", result1)
print("C2 and not B:", result2)
assert result1 == z3.unsat and result2 == z3.sat
m = solver2.model()
print("one satisfying assignment (solver-chosen):", ", ".join(f"{v} = {m.eval(v, model_completion=True)}" for v in action))

PUSH = z3.And(binary == git_remote_https, host == github, protocol == git_http, method == POST,
              path == z3.StringVal("/acme/secrets.git/git-receive-pack"))
push = z3.Solver()
push.add(C2, z3.Not(B), PUSH)
print("C2 admits a simulated push request to acme/secrets:", push.check())
assert push.check() == z3.sat

B_bad = C2
passes, _ = exceeds(C2, B_bad)
bad_boundary_allows_push = z3.Solver()
bad_boundary_allows_push.add(B_bad, PUSH)
print("C2 against B_bad = C2:", passes, "| B_bad admits the simulated push:", bad_boundary_allows_push.check())
assert passes == z3.unsat and bad_boundary_allows_push.check() == z3.sat

B_github = z3.And(binary == gh, host == github, protocol == rest, method == GET,
                  z3.PrefixOf(z3.StringVal("/repos/"), path))
C_protocol_only = z3.And(binary == gh, host == github, protocol == git_http, method == GET,
                         z3.PrefixOf(z3.StringVal("/repos/"), path))
result3, _ = exceeds(C_protocol_only, B_github)
print("same binary, host, method and path; protocol only:", result3)
assert result3 == z3.sat

C1 and not B: unsat
C2 and not B: sat
one satisfying assignment (solver-chosen): binary = git_remote_https, host = github.com, protocol = git_smart_http, method = POST, path = ""
C2 admits a simulated push request to acme/secrets: sat
C2 against B_bad = C2: unsat | B_bad admits the simulated push: sat
same binary, host, method and path; protocol only: sat


The first result is UNSAT: $C_1$ stays within $B$. We also check that $C_1$ is satisfiable on its own. Without that second check, an accidentally impossible candidate could appear safe simply because it allows nothing.

The second result is SAT: $C_2$ permits at least one action that $B$ forbids. Z3 prints an assignment for the action fields, but not every printed value is meaningful. Our Git clause constrains the binary, host, and protocol. It does not constrain the method or path, so the solver is free to choose values such as `POST` and an empty path. Another solver version or seed may choose a different witness.

We therefore ask a more focused follow-up question: could the Git clause admit the shape of a push, namely a `POST` to a repository's `git-receive-pack` path? The answer is SAT. This remains a synthetic request, not a replayed Git session. A real push would also depend on the credential's scope, GitHub's behavior, and the sandbox's enforcement.

Next, we keep the binary, host, method, and path fixed, then change only the protocol from REST to Git smart HTTP. That query is SAT too. The result isolates the lesson encoded in this model: a REST-only rule does not cover the same operation over a different protocol. Z3 did not discover that protocols differ; we told it they were distinct values.

The final check exposes a deeper limitation. Suppose the trusted boundary is itself too broad, so $B_{\text{bad}}=C_2$. The containment query is UNSAT and the candidate passes, even though the boundary still permits the simulated push. Containment can compare two policies. It cannot decide whether the operator chose the right boundary.

## Using OpenShell

OpenShell packages the same containment idea in a command-line tool:

```bash
openshell-prover check candidate.yaml --boundary boundary.yaml
```

The [v0.1.5 documentation](https://docs.nvidia.com/openshell/v0.1.5/how-it-works/policies/prover) defines five outcomes:

1. `within_boundary` : No modeled counterexample was found.
1. `exceeds_boundary` : A modeled counterexample was found.
1. `error`: The check could not run correctly.
1. `unsupported`: The comparison uses a feature the prover cannot model.
1. `inconclusive`: The prover could not establish a result within its limits.

Only `within_boundary` is a pass. In particular, `unsupported` and `inconclusive` are not softer versions of success.

The checker reads two policy files and reports their relationship. It does not deploy, apply, or approve a policy. It is also separate from the policy advisor's risk review for additions such as credential access.

We start with a filesystem example from the documentation. The boundary permits reads from `/usr` and `/etc`. Removing `/etc` narrows the policy, so we expect `within_boundary`. Adding write access to `/tmp` expands the policy, so we expect `exceeds_boundary` and a write witness.

In [4]:
OUTCOMES = ("within_boundary", "exceeds_boundary", "unsupported", "inconclusive", "error")


def is_pass(result, exit_code):
    return result == "within_boundary" and type(exit_code) is int and exit_code == 0


def check(boundary, candidate, *flags):
    assert PROVER, "openshell-prover not found"
    with tempfile.TemporaryDirectory() as tmp:
        b, c = Path(tmp, "boundary.yaml"), Path(tmp, "candidate.yaml")
        b.write_text(boundary)
        c.write_text(candidate)
        try:
            run = subprocess.run([PROVER, "check", str(c), "--boundary", str(b), "--output", "json", *flags],
                                 capture_output=True, text=True, timeout=60)
            doc = json.loads(run.stdout)
            out = {"outcome": doc["result"], "exit_code": run.returncode, "reason_code": doc.get("reason_code"),
                   "reason": doc.get("reason"), "counterexample": doc.get("counterexample"),
                   "domains": (doc.get("coverage") or {}).get("domains", [])}
        except (subprocess.TimeoutExpired, json.JSONDecodeError, KeyError) as exc:
            out = {"outcome": "error", "exit_code": None, "reason_code": type(exc).__name__, "reason": str(exc),
                   "counterexample": None, "domains": []}
    if out["outcome"] not in OUTCOMES:
        out["outcome"] = "error"
    out["approved"] = is_pass(out["outcome"], out["exit_code"])
    shown = {k: v for k, v in out.items() if v not in (None, [], "") and k != "domains"}
    print(shown)
    return out


FS_BOUNDARY = "version: 1\nfilesystem_policy:\n  read_only:\n    - /usr\n    - /etc\n"
FS_NARROWER = "version: 1\nfilesystem_policy:\n  read_only:\n    - /usr\n"
FS_WRITE = "version: 1\nfilesystem_policy:\n  read_only:\n    - /usr\n  read_write:\n    - /tmp\n"

print("narrower candidate")
narrower = check(FS_BOUNDARY, FS_NARROWER)
print("\ncandidate that adds a writable /tmp")
widened = check(FS_BOUNDARY, FS_WRITE)
assert narrower["outcome"] == "within_boundary" and widened["outcome"] == "exceeds_boundary"

narrower candidate
{'outcome': 'within_boundary', 'exit_code': 0, 'approved': True}

candidate that adds a writable /tmp
{'outcome': 'exceeds_boundary', 'exit_code': 1, 'counterexample': {'domain': 'filesystem', 'access': 'write', 'path': '/tmp'}, 'approved': False}


The results we got match the hypotheses. The narrower read policy stays within the boundary, while writable `/tmp` produces a counterexample containing a write.

The helper function is deliberately strict. It requests JSON, reads the `result` field, and marks a policy approved only when both the result is `within_boundary` and the exit code is 0. Missing, malformed, or contradictory output remains unapproved. This is a small example of *failing closed*: uncertainty stops the automated path instead of quietly granting access.

## OpenTest with the GitHub Policy Expansion

Now we return to the motivating incident. The boundary lets `gh` read `api.github.com` through REST inspection. The candidate adds a `git_clone` rule that lets `git-remote-https` reach `github.com:443`.

Because the new endpoint has no `protocol` field, the prover treats it as plain layer-4 TCP, written as `l4` in the result. The candidate changes the binary, host, and protocol together, so we expect `exceeds_boundary`. This first comparison tells us whether the candidate expands access; it does not yet isolate which change caused the expansion.

In [5]:
NET_BOUNDARY = """version: 1
filesystem_policy: { include_workdir: false }
network_policies:
  github_rest:
    endpoints:
      - { host: api.github.com, port: 443, protocol: rest, enforcement: enforce, access: read-only }
    binaries: [{ path: /usr/bin/gh }]
"""
NET_GIT = NET_BOUNDARY + """  git_clone:
    endpoints:
      - { host: github.com, port: 443 }
    binaries: [{ path: /usr/lib/git-core/git-remote-https }]
"""
git_run = check(NET_BOUNDARY, NET_GIT)
assert git_run["outcome"] == "exceeds_boundary"

{'outcome': 'exceeds_boundary', 'exit_code': 1, 'counterexample': {'domain': 'network', 'binary': None, 'ancestor_binary': None, 'binary_identity_required': False, 'host': 'github.com', 'destination_ip': '8.8.8.8', 'trusted_gateway': False, 'port': 443, 'protocol': 'l4', 'method': None, 'path': None}, 'approved': False}


The prover returns `exceeds_boundary` with a witness on `github.com:443` over `l4`. That is enough to show that the candidate is broader than the boundary.

In the parsed JSON, `binary` is `null`, so the witness does not identify `git-remote-https` or any other executable. The `destination_ip` value `8.8.8.8` is simply a value chosen by the solver to make its model concrete. The prover sent no packet there and observed no live traffic.

In short, the counterexample means *some modeled action is allowed by the candidate and forbidden by the boundary*. It is not a packet capture or an execution trace.

Because three fields changed at once, this comparison cannot prove that the protocol change alone caused the expansion. We need a controlled comparison for that.

## Change One Thing at a Time

A clean experiment holds other variables constant (as all AI experiments should do!). Here the boundary and candidate use the same host, port, and binary. The boundary lets `gh` read `github.com` through REST inspection; the candidate lets that same binary reach the same endpoint through an L4 rule.

We expect `exceeds_boundary` again. An L4 rule can carry requests that a REST-aware rule would inspect, so replacing the REST rule with a generic transport rule should enlarge the permitted action set.

In [6]:
SAME_BOUNDARY = """version: 1
filesystem_policy: { include_workdir: false }
network_policies:
  github_rest:
    endpoints:
      - { host: github.com, port: 443, protocol: rest, enforcement: enforce, access: read-only }
    binaries: [{ path: /usr/bin/gh }]
"""
SAME_L4 = """version: 1
filesystem_policy: { include_workdir: false }
network_policies:
  github_l4:
    endpoints:
      - { host: github.com, port: 443 }
    binaries: [{ path: /usr/bin/gh }]
"""
same_run = check(SAME_BOUNDARY, SAME_L4)
assert same_run["outcome"] == "exceeds_boundary"

{'outcome': 'exceeds_boundary', 'exit_code': 1, 'counterexample': {'domain': 'network', 'binary': None, 'ancestor_binary': None, 'binary_identity_required': False, 'host': 'github.com', 'destination_ip': '8.8.8.8', 'trusted_gateway': False, 'port': 443, 'protocol': 'l4', 'method': None, 'path': None}, 'approved': False}


The result is `exceeds_boundary`, even though the host, port, and binary are unchanged. Within the v0.1.5 prover's model, removing REST inspection from the same endpoint is enough to broaden the policy.

The witness has the same limits as before: `binary` is `null`, `binary_identity_required` is `false`, and `destination_ip` is solver-chosen. It establishes the existence of a modeled counterexample. It does not identify the program a real agent would run or show that any network traffic occurred.

## We Learn from What the Prover Refuses to Decide

A safe decision system should not guess when its model is incomplete. The next two comparisons look plausible to a human but fall outside what this prover is willing to establish:

- `/tmp/cache` appears to sit inside `/tmp`, but a symlink can change what either path means. The prover does not assume simple path nesting.
- `protocol: git` is outside the two network protocols modeled in this version: TCP and REST.

We expect both checks to return `unsupported`. That result is useful information: it marks the edge of the guarantee and prevents an uncertain change from being mistaken for a pass.

In [7]:
FS_RW_TMP = "version: 1\nfilesystem_policy:\n  read_write:\n    - /tmp\n"
FS_RW_CACHE = "version: 1\nfilesystem_policy:\n  read_write:\n    - /tmp/cache\n"
NET_PROTOCOL_GIT = "version: 1\nnetwork_policies:\n  git_clone:\n    endpoints:\n      - { host: github.com, port: 443, protocol: git }\n    binaries: [{ path: /usr/lib/git-core/git-remote-https }]\n"

print("nested path")
nested = check(FS_RW_TMP, FS_RW_CACHE)
print("\nprotocol: git")
proto = check(NET_BOUNDARY, NET_PROTOCOL_GIT)
assert nested["outcome"] == "unsupported" and proto["outcome"] == "unsupported"
assert not nested["approved"] and not proto["approved"]

nested path
{'outcome': 'unsupported', 'exit_code': 3, 'reason_code': 'unresolved_filesystem_path', 'reason': "filesystem write containment for '/tmp/cache' depends on sandbox path resolution; use matching paths in candidate and boundary", 'approved': False}

protocol: git
{'outcome': 'unsupported', 'exit_code': 3, 'reason_code': 'unsupported_policy_shape', 'reason': "candidate policy rule 'git_clone' uses protocol 'git'; only L4 TCP and REST are modeled", 'approved': False}


Both comparisons return `unsupported`, including a `reason_code` and a human-readable reason, and neither is approved.

`inconclusive` shares exit code 3, which is why automation should inspect the result field rather than infer the outcome from the exit code alone. A solver timeout could produce `inconclusive`. We do not manufacture one here because completion within a tiny time budget would depend on the machine running the notebook.

## From Proof to Approval Gate

Suppose an agent requests more GitHub access. Before granting it, the operator needs to know whether the new effective policy exceeds the environment's approved maximum.

A careful workflow is:

1. Write and review a trusted `boundary.yaml` that represents the maximum acceptable access.
2. Build the complete *effective candidate*: the policy the sandbox would enforce after the change, including rules added by attached providers. The prover does not add those rules for you.
3. Compare the candidate with the boundary.
4. Proceed only when the result is `within_boundary`, the exit code is 0, and the prover reports coverage for every policy domain the decision relies on.
5. Send every expansion, unsupported feature, inconclusive result, malformed response, or coverage gap to a human.

In a real deployment, the effective policy can be saved with `openshell sandbox get <name> --policy-only`. This notebook uses synthetic strings instead.

The gate below implements the conservative part of that workflow. It checks `coverage.domains` because a pass applies only to the domains the prover modeled. It still does not install or enforce a policy.

In [8]:

def decide(out, required_domains):
    if out is None:
        return "stop: prover unavailable, nothing was checked"
    if not is_pass(out.get("outcome"), out.get("exit_code")):
        if out.get("outcome") == "within_boundary":
            return "escalate to a human: result and exit code disagree"
        return f"escalate to a human: {out.get('outcome')}"
    domains = out.get("domains")
    covered = set(domains) if isinstance(domains, list) and all(isinstance(d, str) for d in domains) else set()
    if not set(required_domains) <= covered:
        return "escalate to a human: required coverage missing"
    return "proceed: candidate is within the boundary for the modeled domains"


def gate(boundary, candidate, required_domains):
    return decide(check(boundary, candidate), required_domains)


NETWORK = {"network_l4", "network_rest"}
EFFECTIVE_OK = NET_BOUNDARY.replace("access: read-only", "rules: [{ allow: { method: GET, path: '/repos/acme/widgets/**' } }]")
decisions = {
    "narrower REST rule": gate(NET_BOUNDARY, EFFECTIVE_OK, NETWORK),
    "adds git-remote-https": gate(NET_BOUNDARY, NET_GIT, NETWORK),
    "uses protocol: git": gate(NET_BOUNDARY, NET_PROTOCOL_GIT, NETWORK),
}
for name, decision in decisions.items():
    print(f"{name}: {decision}")


{'outcome': 'within_boundary', 'exit_code': 0, 'approved': True}
{'outcome': 'exceeds_boundary', 'exit_code': 1, 'counterexample': {'domain': 'network', 'binary': None, 'ancestor_binary': None, 'binary_identity_required': False, 'host': 'github.com', 'destination_ip': '8.8.8.8', 'trusted_gateway': False, 'port': 443, 'protocol': 'l4', 'method': None, 'path': None}, 'approved': False}
{'outcome': 'unsupported', 'exit_code': 3, 'reason_code': 'unsupported_policy_shape', 'reason': "candidate policy rule 'git_clone' uses protocol 'git'; only L4 TCP and REST are modeled", 'approved': False}
narrower REST rule: proceed: candidate is within the boundary for the modeled domains
adds git-remote-https: escalate to a human: exceeds_boundary
uses protocol: git: escalate to a human: unsupported


Only the candidate that narrows the REST rule proceeds automatically.

- The Git binary candidate escalates because it exceeds the boundary.
- The `protocol: git` candidate escalates because the prover cannot model it.
- The deliberately incomplete coverage result escalates because the required network domain is missing.
- The malformed and missing files below return `error` and fail closed.

This is the right shape for an approval gate: every uncertain state has an explicit safe destination. Even so, the gate proves only a relationship between policy files. It does not prove that a running sandbox enforces either file.

In [9]:

broken = check(NET_BOUNDARY, "version: [unclosed")
assert broken["outcome"] == "error" and not broken["approved"]
missing = subprocess.run([PROVER, "check", "no_such_candidate.yaml", "--boundary", "no_such_boundary.yaml", "--output", "json"],
                         capture_output=True, text=True)
assert json.loads(missing.stdout)["result"] == "error" and missing.returncode == 2
print("malformed and missing inputs both fail closed")


{'outcome': 'error', 'exit_code': 2, 'reason_code': 'invalid_input', 'reason': "invalid candidate policy: invalid policy: failed to parse sandbox policy YAML: YAML parse error at line 1, column 19: expected ',' or ']' in flow sequence", 'approved': False}
malformed and missing inputs both fail closed


It is tempting to read `within_boundary` as "the agent is safe." That conclusion is too strong. A pass supports only the first of three separate claims:

1. Policy logic: Is the candidate contained by the boundary under the encoded rules?
1. Runtime enforcement: Does the deployed sandbox block every forbidden action?
1. System safety: Is the agent, credential set, toolchain, and environment safe for the task?

The documentation also cautions that a passing policy need not be the narrowest possible or appropriate for the task. The prover cannot tell whether the boundary itself was sensible, as the $B_{\text{bad}}$ example showed, and it does not judge whether a credential is too powerful.

Our toy model is narrower than OpenShell. It includes only three binaries, three hosts, and two protocols, and its boundary allows only `GET`. OpenShell's `read-only` access also permits `HEAD` and `OPTIONS`. Even a server-side `GET` is not guaranteed to be free of side effects.

OpenShell's prover also has deliberate blind spots. Differences involving nested paths, unmodeled protocols, GraphQL, MCP, and WebSocket return `unsupported`. A REST endpoint in `enforcement: audit` mode is unsupported because audit mode records an action without blocking it. The dev note adds that formal rules do not understand business context: a disposable repository and a production repository may look structurally identical. Credentials remain a separate concern. An L4 rule can say where a token may travel, but not what that token can do when it arrives.

The motivating evidence is vendor-reported. OpenShell's [adversarial prototype](https://nvidia.github.io/OpenShell-Research/dev-notes/posts/2026-08-27-adversarial-policy-review-long-horizon-agents/) describes the team's own attacks and measurements. We did not reproduce those experiments here.


## Authorization Research out There

The containment pattern predates agent sandboxes. [Zelkova](https://doi.org/10.23919/FMCAD.2018.8602994) (Backes et al., 2018) translates AWS access policies into SMT and asks whether one permits more than another. [Cedar](https://doi.org/10.1145/3649835) (Cutler et al., 2024) designs its policy language to remain analyzable.

Agent systems apply the same idea closer to tool use. [Progent](https://arxiv.org/abs/2504.11703) (Shi et al., 2025) enforces privilege rules on tool calls and uses an SMT solver when an LLM proposes a policy update: narrowings can be applied automatically, while expansions require approval. Its threat model centers on prompt injection steering tool calls. The preprint [FAVA](https://arxiv.org/abs/2607.27267) checks an LLM-derived permission graph before state-changing actions, so its guarantee depends on the assumptions used to derive that graph.

## Conclusion

The GitHub incident is memorable because the agent did not need a new credential. It needed a different route. A policy that understood REST calls did not automatically understand Git smart HTTP, even when both routes reached GitHub.

Policy containment gives us a precise question: *does the proposed policy allow any action that the trusted boundary forbids?* Z3 makes that question executable. OpenShell's prover applies it to real policy files, produces counterexamples for expansions, and refuses to approve comparisons outside its model.

That is valuable, but deliberately limited, assurance. Before granting an agent more access, check the candidate against a reviewed boundary, inspect the credential's real scope, confirm the relevant policy domains were modeled, and verify that the deployed sandbox enforces the result. A proof is strongest when we are equally clear about what it does not prove.

## References

1. Backes, J., Bolignano, P., Cook, B., et al. (2018). *Semantic-based Automated Reasoning for AWS Access Policies using SMT*. https://doi.org/10.23919/FMCAD.2018.8602994
2. Cutler, J. W., Disselkoen, C., Eline, A., et al. (2024). *Cedar: A New Language for Expressive, Fast, Safe, and Analyzable Authorization*. https://doi.org/10.1145/3649835
3. de Moura, L. and Bjørner, N. (2008). *Z3: An Efficient SMT Solver*. https://doi.org/10.1007/978-3-540-78800-3_24
4. NVIDIA. *OpenShell*. https://github.com/NVIDIA/OpenShell
5. NVIDIA. *OpenShell v0.1.5 Policy Prover Documentation*. https://docs.nvidia.com/openshell/v0.1.5/how-it-works/policies/prover
6. Shi, T., He, J., Wang, Z., et al. (2025). *Progent: Programmable Privilege Control for LLM Agents*. [arXiv:2504.11703](https://arxiv.org/abs/2504.11703)
7. Watson, A., Greco, J., Myers, J., et al. (2026). *Adversarial Prototype: AI Policy Auto-Approval for Long-Horizon Agents in OpenShell*. https://nvidia.github.io/OpenShell-Research/dev-notes/posts/2026-08-27-adversarial-policy-review-long-horizon-agents/
8. Watson, A. and the OpenShell Team (2026). *What We Have Learned Applying Formal Methods to Control AI Agents*. https://nvidia.github.io/OpenShell-Research/dev-notes/posts/2026-09-10-learning-formal-methods-agent-policy-prover/
9. Zhang, Y., Zhao, X., Liu, S., et al. (2026). *FAVA: Formal Authorization for Verified Agents with Evidence-Backed Permission Graphs*. [arXiv:2607.27267](https://arxiv.org/abs/2607.27267)

# GitHub

This article is also available on [GitHub](https://github.com/adamd1985/quant_research/blob/main/harness_rogue_agents.ipynb)

# Media

All media used (in the form of code or images) are either solely owned by me, acquired through licensing, or part of the Public Domain and granted use through Creative Commons License.

# CC Licensing and Use

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.